# quantbench_paper_soxl_v1

Notebook-first evidence surface for adapting QuantBench (arXiv:2504.18600) to the SOXL harness.

The architecture logic lives in `src.soxl_quant.app.quantbench_screen`; this notebook only executes that logic and records the resulting evidence.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "quantbench_paper_soxl_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.quantbench_screen import QuantBenchGate, build_quantbench_screen_report
from src.soxl_quant.app.quantbench_supervised import build_quantbench_supervised_report
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

print("root=.")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model_selection/card/quantbench_paper_soxl_v1


## Execute Src-Owned QuantBench Screen

The strict gate encodes the user target: full test-proxy MaxDD <= 20%, positive proxy return, strong 2022 benchmark-relative defense, nonzero stress exposure so the strategy retains chips, and at least 50% capture of 2023-2024 buy-and-hold upside.

In [2]:
gate = QuantBenchGate(
    max_drawdown_pct=20.0,
    min_stress_relative_return=30.0,
    min_stress_avg_pos=0.10,
    max_stress_avg_pos=0.45,
    min_bull_capture=0.50,
    min_proxy_return=0.0,
)
report = build_quantbench_screen_report(root=ROOT, card_dir=CARD, gate=gate)
print(json.dumps({
    "decision": report["decision"],
    "records": report["screen"]["records"],
    "eligible_profiles": report["screen"]["eligible_profiles"],
}, indent=2))

{
  "decision": "no_current_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 26,
  "eligible_profiles": []
}


## Execute Supervised QuantBench Models

This section trains the paper-style supervised model/objective families through `src.soxl_quant.app.quantbench_supervised` on the existing SOXL feature architecture, including macro-risk and relation packs.

In [3]:
feature_packs = (
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation",
)

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=feature_packs, profile_name="quantbench_paper_soxl_v1")
features = dataset.handler.feature_frame
close = dataset.handler.price_frame["price"]

supervised_report = build_quantbench_supervised_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    feature_packs=feature_packs,
    gate=gate,
)
print(json.dumps({
    "decision": supervised_report["decision"],
    "records": supervised_report["screen"]["records"],
    "eligible_profiles": supervised_report["screen"]["eligible_profiles"],
    "feature_count": supervised_report["data"]["feature_count"],
}, indent=2))

[55181:MainThread](2026-06-06 22:35:30,365) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[55181:MainThread](2026-06-06 22:35:30,775) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[55181:MainThread](2026-06-06 22:35:30,775) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/.worktrees/quantbench-paper-soxl-v1/data/qlib/soxl')}


[55181:MainThread](2026-06-06 22:35:32,486) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[55181:MainThread](2026-06-06 22:35:32,487) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[55181:MainThread](2026-06-06 22:35:32,487) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{
  "decision": "no_supervised_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 7,
  "eligible_profiles": [],
  "feature_count": 113
}


## Supervised Model / Objective Screen

In [4]:
supervised_specs_df = pd.read_csv(CARD / "supervised_model_specs.csv")
supervised_metrics_df = pd.read_csv(CARD / "supervised_objective_metrics.csv")
supervised_screen_df = pd.read_csv(CARD / "supervised_candidate_screen.csv")
print({
    "eligible": int(supervised_screen_df["eligible"].sum()) if not supervised_screen_df.empty else 0,
    "rows": len(supervised_screen_df),
})
display(supervised_specs_df)
display(supervised_metrics_df)
display(supervised_screen_df.head(20))

{'eligible': 0, 'rows': 7}


,name,paper_family,objective,target_kind,estimator
0,logistic_clf,linear_classifier,clf,binary,LogisticRegression
1,ridge_ic,linear_regression,ic,return,Ridge
2,ridge_mse,linear_regression,mse,return,Ridge
3,random_forest_clf,tree,clf,binary,RandomForestClassifier
4,hist_gradient_mse,boosted_tree,mse,return,HistGradientBoostingRegressor
5,extra_trees_rank,tree_ensemble,ranking,return,ExtraTreesRegressor
6,mlp_mse,neural_tabular,mse,return,MLPRegressor


,model_name,paper_family,objective,rows,clf_accuracy,ic,mse,ranking_spread,utility_return
0,logistic_clf,linear_classifier,clf,1594.0,0.471142,0.084410,0.267675,0.007406,0.018179
1,ridge_ic,linear_regression,ic,1594.0,0.444793,-0.110690,0.317109,-0.007968,0.009132
2,ridge_mse,linear_regression,mse,1594.0,0.444793,-0.110690,0.317109,-0.007968,0.009132
3,random_forest_clf,tree,clf,1594.0,0.500000,-0.032808,0.104747,-0.074851,0.008161
4,hist_gradient_mse,boosted_tree,mse,1594.0,0.456713,0.049200,0.111894,0.054097,0.010012
5,extra_trees_rank,tree_ensemble,ranking,1594.0,0.493726,-0.017903,0.099501,-0.003517,0.007681
6,mlp_mse,neural_tabular,mse,1594.0,0.484944,-0.012545,0.274140,0.003873,0.032648


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,quantbench_supervised_mlp_mse,generated:quantbench_supervised_mlp_mse,False,1353.355757,"['max_drawdown', 'stress_avg_pos', 'stress_max...",1498.768596,67.409087,63.676846,51.531111,0.498757,231.176563,191.773509,1.205467
1,quantbench_supervised_ridge_ic,generated:quantbench_supervised_ridge_ic,False,724.062318,"['max_drawdown', 'stress_avg_pos', 'stress_max...",854.489316,68.178628,70.426229,50.351475,0.498452,252.875013,191.773509,1.318613
2,quantbench_supervised_ridge_mse,generated:quantbench_supervised_ridge_mse,False,724.062318,"['max_drawdown', 'stress_avg_pos', 'stress_max...",854.489316,68.178628,70.426229,50.351475,0.498452,252.875013,191.773509,1.318613
3,quantbench_supervised_logistic_clf,generated:quantbench_supervised_logistic_clf,False,611.587891,"['max_drawdown', 'stress_avg_pos', 'stress_max...",728.500036,66.516879,50.332270,53.015178,0.498300,304.580636,191.773509,1.588231
4,quantbench_supervised_hist_gradient_mse,generated:quantbench_supervised_hist_gradient_mse,False,405.901101,"['max_drawdown', 'stress_avg_pos', 'stress_max...",681.761273,70.883673,40.881526,57.146925,0.498427,89.736570,191.773509,0.467930
5,quantbench_supervised_random_forest_clf,generated:quantbench_supervised_random_forest_clf,False,367.879856,"['max_drawdown', 'stress_avg_pos', 'stress_max...",587.418900,65.661613,44.644049,53.546039,0.498186,112.117238,191.773509,0.584634
6,quantbench_supervised_extra_trees_rank,generated:quantbench_supervised_extra_trees_rank,False,53.897026,"['max_drawdown', 'stress_avg_pos', 'stress_max...",325.009533,78.137709,48.887293,49.873473,0.499303,100.778966,191.773509,0.525510


## Paper Requirement Matrix

QuantBench is a full benchmark architecture: data preparation, factor mining, modeling, portfolio/execution objectives, and task-aware evaluation. The requirement matrix is emitted by src and persisted as a sidecar CSV.

In [5]:
requirements_df = pd.read_csv(CARD / "paper_requirement_matrix.csv")
display(requirements_df)

,key,paper_basis,soxl_mapping,next_artifact
0,data_preparation,"Unified data preparation with breadth, depth, ...","SOXL/SOXX/VIX daily panel, qlib binary store, ...",runtime/data or runtime/model_selection notebo...
1,factor_mining,Formula-based factor mining and learned factor...,"Feature pack diagnostics with IC, standalone b...",runtime/factor/card/<pack>/<pack>.ipynb plus f...
2,modeling_tree,Tree models can outperform deep nets on strong...,Add non-RL tabular baselines such as XGBoost/L...,runtime/model_selection/card/quantbench_paper_...
3,modeling_sequence,Temporal models such as LSTM are benchmarked a...,Existing PPO actor surfaces include MLP fallba...,network/profile matrix and smoke/full retrain ...
4,modeling_relational,Relational or graph models test whether market...,Single-SOXL setting maps this to SOXL/SOXX/VIX...,SOXX-relative and multi-asset relation feature...
5,training_objectives,"Classification, IC, MSE, ranking, utility, and...","Compare return-prediction labels, IC/ranking p...",objective-family screen with fixed-split and w...
6,rolling_continual_learning,Rolling updates mitigate alpha decay and distr...,Use existing walk-forward folds plus 3/6/12-mo...,runtime/model_selection rolling cadence notebo...
7,ensemble_robustness,Seed ensembles reduce low signal-to-noise over...,Aggregate low-correlated actors/signals only a...,ensemble candidate summary with pairwise corre...
8,evaluation_metrics,"Evaluation should include signal, portfolio, e...","Gate on IC/ICIR, return, Sharpe, MaxDD, turnov...",quantbench_screen summary.json plus candidate ...


## Model / Objective Coverage Ledger

This ledger is a coverage claim, not a performance claim. Missing families are explicit implementation tasks.

In [6]:
coverage_df = pd.read_csv(CARD / "paper_model_objective_coverage.csv")
display(coverage_df)

,family,paper_family,current_soxl_surface,coverage,next_step
0,tree_xgboost_lightgbm,XGBoost/tree model,sklearn tree/boosted supervised baselines impl...,implemented_partial,promote only if QuantBench gate passes; otherw...
1,sequence_lstm_gru,LSTM/temporal sequence,GRU actor exists; LSTM supervised objective mi...,partial,compare GRU PPO actor to supervised sequence l...
2,attention_tide_mixer,deep sequence / mixer style,attention and TiDE actor surfaces exist; MLP-M...,partial,small-capacity mixer/attention screen only aft...
3,relational_rgcn,RGCN/relational model,SOXX-relative and multi-asset relation feature...,partial,screen relation features before graph implemen...
4,objectives_clf_ic_mse_ranking,CLF / IC / MSE / Ranking,common supervised target builder and CLF/IC/MS...,implemented,use objective metrics and risk-gate failures t...
5,utility_rl_execution,utility maximization / RL / execution,"PPO reward, risk budget, execution overlays exist",implemented_partial,use QuantBench gate for promotion selection
6,rolling_continual,3/6/12 month rolling updates,walk-forward folds exist; cadence-specific rol...,partial,add rolling cadence screen with cost/evidence ...
7,seed_ensemble,seed ensemble for overfit mitigation,single-profile summaries exist; no general ens...,missing,rank low-correlation candidates and ensemble o...


## Objective Family Mapping

This table is emitted by `src.soxl_quant.app.quantbench_objectives` and defines the shared SOXL target semantics for paper-style CLF, IC, MSE, ranking, utility, and RL comparisons.

In [7]:
objective_df = pd.read_csv(CARD / "paper_objective_families.csv")
display(objective_df)

,key,paper_basis,soxl_target,metric
0,clf,Classification loss predicts future up/down mo...,fwd_positive_h,clf_accuracy
1,ic,IC objective aligns predictions with future re...,fwd_return_h,ic
2,mse,MSE loss regresses future returns directly.,fwd_return_h,mse
3,ranking,Pairwise or listwise ranking objectives prefer...,relative ordering of fwd_return_h,ranking_spread
4,utility,Utility maximization should reflect trading ou...,position-weighted fwd_return_h with chip reten...,utility_return
5,rl,Non-differentiable simulation and execution fe...,PPO reward and execution gate metrics,portfolio_return_sharpe_maxdd


## Current Runtime Candidate Screen

The ranking scans current runtime summaries through the src-owned QuantBench gate. It does not retrain models in this notebook.

In [8]:
screen_df = pd.read_csv(CARD / "candidate_screen.csv")
print({"eligible": int(screen_df["eligible"].sum()) if not screen_df.empty else 0, "rows": len(screen_df)})
display(screen_df.head(20))

{'eligible': 0, 'rows': 26}


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,balanced_rule_residual_v19_reg_v22_retest_v1,runtime/model/card/balanced_rule_residual_v19_...,False,1119.521092,"['max_drawdown', 'stress_max_drawdown']",1280.416932,59.637288,37.652465,58.299843,0.380930,44.524050,55.654474,0.800008
1,balanced_rule_residual_v22_continued_retention_v1,runtime/model/card/balanced_rule_residual_v22_...,False,1015.381197,"['max_drawdown', 'stress_max_drawdown']",1128.935617,59.130395,53.736876,55.255626,0.321447,60.791539,55.654474,1.092303
2,balanced_rule_residual_v22_release_v1,runtime/model/card/balanced_rule_residual_v22_...,False,1002.849540,"['max_drawdown', 'stress_max_drawdown']",1133.887381,60.350126,45.981620,38.845207,0.375263,58.092719,55.654474,1.043810
3,balanced_rule_residual_v19_moe_v2,runtime/model/card/balanced_rule_residual_v19_...,False,610.398973,"['max_drawdown', 'stress_max_drawdown']",690.971221,49.008279,47.586050,45.149295,0.287632,60.037161,55.654474,1.078748
4,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,False,559.292883,"['max_drawdown', 'stress_max_drawdown']",619.708826,43.565118,52.906964,41.483544,0.258405,56.176271,55.654474,1.009376
5,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,False,426.778414,"['max_drawdown', 'stress_max_drawdown']",496.780712,44.234213,65.015154,43.909386,0.131381,45.591811,55.654474,0.819194
6,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,False,351.838655,"['max_drawdown', 'stress_max_drawdown']",392.580698,37.676486,55.990314,35.878546,0.185465,52.300504,55.654474,0.939736
7,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,False,-1032.939429,"['proxy_return', 'bull_capture']",-7.000281,9.875679,53.563568,9.875679,0.165855,NaN,NaN,NaN
8,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,False,-1054.613515,"['proxy_return', 'bull_capture']",-12.705282,15.900697,61.694557,15.900697,0.141487,NaN,NaN,NaN
9,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,False,-1174.741020,"['max_drawdown', 'proxy_return', 'stress_max_d...",-23.275008,29.147985,45.125927,29.147985,0.243813,NaN,NaN,NaN


## Conclusion

This notebook now executes two src-owned QuantBench-to-SOXL architecture layers: existing runtime-summary candidate screening and a supervised paper-style model/objective screen trained on the repo's SOXL feature frame. The supervised run includes macro-risk, academic, rulebase, transition, VIX-term, and SOXX-relation feature packs.

Use `summary.json` for the current runtime-model screen and `supervised_summary.json` for the new supervised architecture screen. A model is promotable only if the relevant summary reports an eligible candidate under MaxDD < 20%, stress chip retention, and bull-market capture gates. If both summaries report no eligible candidate, the paper-backed next step is rolling/ensemble validation or a new macro-risk feature experiment, not promotion.